## Базовый запуск и инференс модели

В ultralytics за инференс отвечает метод model.predict() (или просто model(), это шорткат).

Когда ты вызываешь results = model.predict(...), функция возвращает список объектов Results. В списке столько элементов, сколько картинок (или кадров видео) ты скормил модели за один раз.

Фактически results это массив выходных кадров, в нашем случае мы передаем картинку одно, но можно передать массив, и они также будут возвращены в массиве result

Из чего состоит объект Result:

Главыне параметры:
- result.path - путь к исходному файлу (изображению)
- result.names - словарь с названиями классов `{0: 'person', 1: 'bicycle'}`
- result.boxes - объект, содержащий все bounding boxes (найденые рамки)

result.boxes - это тензоры PyTorch, что бы перевести их в обычные числа, нужно использовать `.cpu().numpy()` или `.tolist()`

result.boxes.xyxy - координаты рамки в формате `[x_min, y_min, x_max, y_max]` в пикселях
result.boxes.xywh - координаты в формате `[x_center, y_center, width, height]`
result.boxes.conf — уверенность модели (confidence score) от 0 до 1.
result.boxes.cls - идентификатор класса (например 0 для person)
result.boxes.id - ID трека (если используется traking)

### Главные параметры model.predict()

#### Фильтрация и точность:
- conf (float, по умолчанию 0.25) - порог уверености. Отбрасывает все объекты, в которых модель уверена мешьне этого значения. Рекомендуется ставить 0.4 до 0.6 для продакшена.
- iou (float, по умолчанию 0.7) - порог IoU для NMS (Non-Maximum Suppression). Помогает убрать дубликаты рамок, которые попали на один и тот же объект
- classes (list[int]) - оставить только нужные классы. Например, classes=[0, 2] оставит только людей и машины, игнорируя всё остальное.

#### Сохранение и вывод
- save (bool, по умолчанию False) - сохранить картинки с нарисованными рамками на диск.
- save_text (bool) - сохранить результаты в текстовый файл (в формате YOLO)
- save_crop (bool) - сохранить каждую найденую рамку как отдельную обрезаную картинку (полезно для сбора датасета)
- show (bool) - открыть окно и показать результат в релаьном времени (использует OpenCV)
- project (str) и name (str) - куда сохранять. project='runs/detect', name='my_exp'. Итоговый путь будет runs/detect/my_exp.
- exist_ok (bool) - если папка с таким именим уже есть, перезаписать ее, иначе создастся папка с приставкой (порядковый номер)

#### Производительность и железо
- device - на чем считать: 'cpu', '0' (первая GPU), ['0', '1'] (несколько GPU)
- half (bool) - использовать FP16 (половину точности). Ускоряет инференс на GPU в 1.5 - 2 раза, без потери качества
- imgsz (int) — до какого размера ресайзить картинку перед подачей в сеть. По умолчанию 640. Важно: для инференса можно ставить любой размер, но кратный 32.

Как автоматически узнать свой девайс (так как мы на маках, то у нас можно просто указывать mps), но также автоматически это можно сделать через PyTorch

```python
import torch

torch.cuda.is_available() -> bool
torch.backends.mps.is_available() -> bool
```

### Сохранение в файл через код

Через OpenCV - для этого можно использовать встроенный мето `plot()`. Также можно поменять толщину линий и размер текста передав в `plot()` line_width и font_size.

```python
import cv2

annotated_image = results[0].plot()

cv2.imwrite('my_result_image.jpg', annotated_image)
```

### Важно при работе с видео (не поток)

Если на вход подается видео например `video.mp4` то predict не вернет список результатов для всего вдео сразу, так как это скорее всего забьет память. Он будет возвращать Result для каждого кадра по очереди (если использовать генератор).

```python
import cv2
from ultralytics import YOLO

results = YOLO('yolo11n.pt')

results = model.predict(source='video.mp4', stream=True, show=True) 

for result in results:
    pass 
```

Параметр stream=True важен что бы не грузить все в память

## Пример инференса модели

In [5]:
from ultralytics import YOLO

model = YOLO('weights/hazard/yolo26s.pt')

results = model('dataset/photo4.jpg')

print(results[0])


image 1/1 /Users/egormuhaev/Documents/Проекты/monitoring-construction-site/dataset/photo4.jpg: 384x640 2 machinerys, 3 vehicles, 96.4ms
Speed: 6.4ms preprocess, 96.4ms inference, 6.3ms postprocess per image at shape (1, 3, 384, 640)
ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
depth: None
keypoints: None
masks: None
names: {0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'utility pole', 10: 'vehicle'}
obb: None
orig_img: array([[[195, 176, 168],
        [195, 176, 168],
        [195, 176, 168],
        ...,
        [224, 209, 206],
        [224, 209, 206],
        [224, 209, 206]],

       [[195, 176, 168],
        [195, 176, 168],
        [195, 176, 168],
        ...,
        [224, 209, 206],
        [224, 209, 206],
        [224, 209, 206]],

       [[196, 177, 169],
        [196, 177, 169],
        [196, 177, 169],
        ..

## Обучение модели

Суть в том что мы не берем и не обучаем модель с нуля в YOLO. Мы берем готовую модель которая уже обучена на датасете COCO (330к изображения с 80 базовыми классами) - модель уже умеет различать объекты, текстуру, контуры. Мы дообучаем ее с использованием технологии **Transfer Learning**.

### Что происходит за одну эпоху

Этап 1:

- Берем батч из 16 случайных картинок
- Применяем аугментации (повороты, яркость, mosaic)
- Ресайзим до 640x640 

Этап 2: Прямой проход

- Картинка проходит через нейросеть
- На выходе модель предсказывает:
  - Координаты рамок (bbox)
  - Уверенность (confidence)
  - Класс объекта (class)  

Этап 3: Подсчет ошибок

- Сравниваем предсказания с разметкой (ground truth)
- Считаем, насколько модель ошиблась 

Этап 4: Обратный проход

- Считаем градиенты — «в каком направлении менять веса, чтобы уменьшить ошибку»
- Используется алгоритм Backpropagation

Этап 5: Обновление весов

- Оптимизатор (обычно AdamW) корректирует веса  
- Модель становится чуть лучше


### Функция потерь (Loss)

YOLO считает ошибку по трем компонентам:

- Box Loss - ошибка рамок. На сколько точно модель предсказала координаты [x_center, y_center, width, height]. Для оценки используется продвинутая метрика CIoU. Она учитывает не только перекрытие рамок, но и расстояние между ними и соотношение сторон.
- Confidence Loss - ошибка уверености. На сколько точно модель оценивает, что в рамке вообще есть объект. Отделяет что "Это человек с увереностью 0.9" от "Это просто фон, но модель думает что это человек"
- Class Loss - ошибка класса. Насколько точно модель определила класс объекта (человек vs машина vs дрон). Используется BCE Loss (Binary Cross-Entropy)

**Итоговый Loss = Box Loss + Confidence Loss + Class Loss**

В логах обучения будут отображаться следующие поля:
- Epoch (эпоха обучения)
- GPU (номер gpu)
- box_loss   
- obj_loss   
- cls_loss
- Instances
- Picture 

### Аргументация

Это одна из основных фишек YOLO - искуственное расширение датасета, за счет применения различнхы преобразований изображения путем искажения исходных.

Аргументы:

1. Mosaic - Склеивает 4 картинки в одну - Учит видеть объекты в разных контекстах
2. MixUp - Смешивает 2 картинки с прозрачностью - Повышает робастность
3. HSV - Меняет оттенок, насыщенность, яркость - Учит работать при разном освещении
4. Flip (LR/UD) - Отзеркаливает по горизонтали/вертикали - Удваивает датасет
5. Rotate - Поворачивает на случайный угол - Учит распознавать под разными углами
6. Scale - Меняет размер - Учит видеть объекты крупным планом и издалека
7. Erasing - Случайно затирает часть картинки - Учит не зависеть от отдельных деталей

### Метрики

Как мы уже упомиали ранее, есть функции потерь - их три. Фактически это значения которые модель пытается минимизировать с каждой эпохой. Но они в основном требуются для оптимизации весов самой нейросети

Метрики же показывают на сколько хорошо реально ищет объекты.

YOLO это модель детекци, и ее задача определить три вещи: 
- Где находится объект (box рамка)
- Класс объекта 
- На сколько модель уверена в этом

#### Ground Truth: что считается правильным ответом

Во время обучения у модели уже есть правильные ответы - они и называеются ground truth или разметка. Обычно координаты нормируются от 0 до 1 относительно ширины и высоты изображения.

#### Базовые понятия

Bounding box — это прямоугольная рамка вокруг объекта.

Confidence — уверенность модели.

IoU — это мера перекрытия предсказанной рамки и правильной рамки. IoU = площадь пересечения рамок / площадь объединения рамок

0.00 - рамки вообще не пересекаются
0.20 - слабое перекрытие
0.50 - среднее перекрытие, часто используется как порог
0.75 - хорошее перекрытие
0.95 - почти идеальное совпадение
1.00 - полное совпадение

Отсюда появляются метрики:
- mAP50
- mAP75
- mAP50-95

#### NMS - Non-Maximum Suppression

Модель может выдать несколько рамок вокруг одного и того же объекта. Например, 5 рамок вокруг одной кошки. NMS убирает дубли.
Он работает примерно так:
- Берет предсказания с наибольшей уверенностью.
- Смотрит, есть ли рядом другие рамки с высоким перекрытием.
- Удаляет лишние дубли.

У NMS тоже есть порог перекрытия, обычно в диапазоне: 0.45 до 0.7

### Как формируются метрики обучения: общая последовательность

Модель получает изображение, а на выходе получается много кандидатов, например:

class: person, confidence: 0.91, box: ...
class: person, confidence: 0.84, box: ...
class: car, confidence: 0.62, box: ...
class: car, confidence: 0.17, box: ...

Далее применяется порог уверености, и объекты которые имеют увереность ниже указаной отсеиваются.
